# 10_Deeper_PbI_VQE_Honest_Milestone_ReTest_Nitrogen_Transfer
## Real executable code — Materia Arche V3.2
### Strategic decision: what to do after 6 failed quantum experiments

NB08 tested 3 circuit variants (fixed). NB09 tested 2 trained variants.
All 6 experiments showed negative quantum lift. This notebook:
1. Tests one final approach (VQE-inspired per-device energy feature)
2. Makes a strategic decision based on accumulated evidence
3. Re-evaluates milestones with full honesty

In [1]:
!pip install pandas numpy pennylane scikit-learn scipy -q

import pandas as pd
import numpy as np
import pennylane as qml
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from scipy.stats import kendalltau
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded")

zsh:1: command not found: pip


/Users/johnodowd/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✅ Libraries loaded


In [2]:
# Load data and re-establish baselines
df = pd.read_csv("perovskite_stability_clean.csv")
print(f"Loaded {len(df)} devices")

ml_features = ['Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
               'MA', 'FA', 'Cs',
               'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
               'Perovskite_thickness', 'Cell_area_measured', 'JV_default_Voc',
               'JV_default_Jsc', 'JV_default_FF']

X_ml = df[ml_features].fillna(0)
y = np.log1p(df['Stability_PCE_T80'])

X_train, X_test, y_train, y_test = train_test_split(X_ml, y, test_size=0.2, random_state=42)

rf_ml = RandomForestRegressor(n_estimators=200, random_state=42)
rf_ml.fit(X_train, y_train)
pred_ml = rf_ml.predict(X_test)
tau_ml, _ = kendalltau(y_test, pred_ml)
mae_ml = mean_absolute_error(y_test, pred_ml)
print(f"ML baseline: tau-b = {tau_ml:.4f}, MAE = {mae_ml:.4f}")

Loaded 1543 devices


ML baseline: tau-b = 0.2490, MAE = 1.3053


## 1. Final quantum experiment: VQE-inspired per-device energy

Previous approaches encoded composition as rotation angles.
This approach: use a small VQE-style circuit where each device's
composition defines a **Hamiltonian** (not just rotations), and we
compute a variational ground-state energy per device.

This is closer to real quantum chemistry — the circuit optimizes
for each device's specific composition.

In [3]:
print("=" * 65)
print("EXPERIMENT: Per-device VQE energy feature")
print("=" * 65)

n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

def build_hamiltonian(row):
    """Build a simple Hamiltonian from device composition.
    Uses Pb, I, Br fractions as coupling constants.
    This models interatomic interactions, not just rotations."""
    pb = float(row['Pb']) if pd.notna(row['Pb']) else 0.0
    i_frac = float(row['I']) if pd.notna(row['I']) else 0.0
    br = float(row['Br']) if pd.notna(row['Br']) else 0.0
    bg = float(row['Perovskite_band_gap']) if pd.notna(row['Perovskite_band_gap']) else 1.5
    
    # Hamiltonian: H = -pb*Z0 - i*Z1 - br*Z2 - bg/3*Z3
    #              + pb*i * Z0Z1 + i*br * Z1Z2 + pb*bg/3 * Z0Z3
    coeffs = [
        -pb, -i_frac, -br, -bg/3.0,
        pb * i_frac, i_frac * br, pb * bg / 3.0
    ]
    obs = [
        qml.PauliZ(0), qml.PauliZ(1), qml.PauliZ(2), qml.PauliZ(3),
        qml.PauliZ(0) @ qml.PauliZ(1),
        qml.PauliZ(1) @ qml.PauliZ(2),
        qml.PauliZ(0) @ qml.PauliZ(3)
    ]
    return qml.Hamiltonian(coeffs, obs)

def vqe_energy(row, n_steps=15):
    """Run a small VQE optimization for this device's Hamiltonian."""
    H = build_hamiltonian(row)
    
    @qml.qnode(dev)
    def circuit(params):
        for i in range(n_qubits):
            qml.RY(params[i], wires=i)
        qml.CNOT(wires=[0, 1])
        qml.CNOT(wires=[2, 3])
        for i in range(n_qubits):
            qml.RY(params[i + n_qubits], wires=i)
        qml.CNOT(wires=[1, 2])
        qml.CNOT(wires=[3, 0])
        return qml.expval(H)
    
    # Quick VQE optimization
    params = np.zeros(2 * n_qubits)
    opt = qml.GradientDescentOptimizer(stepsize=0.4)
    for _ in range(n_steps):
        params = opt.step(circuit, params)
    
    return float(circuit(params))

# Run VQE for all devices (this takes a while)
print(f"Running per-device VQE ({len(df)} devices, 15 steps each)...")
print("This will take several minutes.")
vqe_energies = []
for idx, row in df.iterrows():
    e = vqe_energy(row)
    vqe_energies.append(e)
    if (idx + 1) % 500 == 0:
        print(f"  Processed {idx + 1}/{len(df)} devices")

vqe_energies = np.array(vqe_energies)
print(f"\nVQE energy range: [{vqe_energies.min():.4f}, {vqe_energies.max():.4f}]")
print(f"Unique values: {len(np.unique(np.round(vqe_energies, 6)))}")

# Check correlation with target
valid_mask = ~np.isnan(vqe_energies)
if valid_mask.sum() == len(vqe_energies):
    corr = np.corrcoef(vqe_energies, y.values)[0, 1]
    tau_vqe, p_vqe = kendalltau(vqe_energies, y.values)
    print(f"Correlation with log(T80): r = {corr:.4f}, tau = {tau_vqe:.4f} (p = {p_vqe:.2e})")
else:
    print(f"Warning: {(~valid_mask).sum()} NaN energies")
    vqe_energies = np.nan_to_num(vqe_energies, nan=0.0)

EXPERIMENT: Per-device VQE energy feature
Running per-device VQE (1543 devices, 15 steps each)...
This will take several minutes.


  Processed 500/1543 devices


  Processed 1000/1543 devices


  Processed 1500/1543 devices

VQE energy range: [-13.5267, 10648.5000]
Unique values: 103
Correlation with log(T80): r = -0.0063, tau = 0.0349 (p = 5.91e-02)


In [4]:
# Evaluate VQE energy as a feature
X_vqe = X_ml.copy().reset_index(drop=True)
X_vqe['vqe_energy'] = vqe_energies

X_train_v, X_test_v, _, _ = train_test_split(X_vqe, y, test_size=0.2, random_state=42)

rf_vqe = RandomForestRegressor(n_estimators=200, random_state=42)
rf_vqe.fit(X_train_v, y_train)
pred_vqe = rf_vqe.predict(X_test_v)
tau_vqe_model, _ = kendalltau(y_test, pred_vqe)
mae_vqe = mean_absolute_error(y_test, pred_vqe)

print("=" * 65)
print("VQE ENERGY FEATURE RESULTS")
print("=" * 65)
print(f"ML-only:          tau-b = {tau_ml:.4f}, MAE = {mae_ml:.4f}")
print(f"ML + VQE energy:  tau-b = {tau_vqe_model:.4f}, MAE = {mae_vqe:.4f}")
print(f"Quantum lift:     {tau_vqe_model - tau_ml:+.4f}")

VQE ENERGY FEATURE RESULTS
ML-only:          tau-b = 0.2490, MAE = 1.3053
ML + VQE energy:  tau-b = 0.2439, MAE = 1.3060
Quantum lift:     -0.0051


## 2. Complete experiment summary (NB02 + NB08 + NB09 + NB10)

In [5]:
vqe_lift = tau_vqe_model - tau_ml

print("=" * 75)
print("ALL QUANTUM EXPERIMENTS — COMPLETE HISTORY")
print("=" * 75)
print(f"{'NB':<5} {'Experiment':<40} {'lift':>8}")
print("-" * 55)
print(f"{'02':<5} {'Fixed 4q RY+CNOT (v1)':<40} {-0.0190:>+8.4f}")
print(f"{'08':<5} {'Fixed 6q 2-layer (v2)':<40} {-0.0097:>+8.4f}")
print(f"{'08':<5} {'ZZ interaction observables':<40} {-0.0024:>+8.4f}")
print(f"{'08':<5} {'GBR + v1 quantum':<40} {-0.0276:>+8.4f}")
print(f"{'08':<5} {'GBR + v2 quantum':<40} {-0.0364:>+8.4f}")
print(f"{'08':<5} {'GBR + ZZ interactions':<40} {-0.0138:>+8.4f}")
print(f"{'09':<5} {'Trained single observable':<40} {-0.0098:>+8.4f}")
print(f"{'09':<5} {'Trained multi-observable':<40} {-0.0099:>+8.4f}")
print(f"{'10':<5} {'Per-device VQE energy':<40} {vqe_lift:>+8.4f}")
print("-" * 55)

all_lifts = [-0.0190, -0.0097, -0.0024, -0.0276, -0.0364, -0.0138,
             -0.0098, -0.0099, vqe_lift]
best_lift = max(all_lifts)
any_positive = any(l > 0 for l in all_lifts)

print(f"\nBest lift across 9 experiments: {best_lift:+.4f}")
print(f"Experiments with positive lift: {sum(1 for l in all_lifts if l > 0)}/9")
print(f"Target: +0.1500")

ALL QUANTUM EXPERIMENTS — COMPLETE HISTORY
NB    Experiment                                   lift
-------------------------------------------------------
02    Fixed 4q RY+CNOT (v1)                     -0.0190
08    Fixed 6q 2-layer (v2)                     -0.0097
08    ZZ interaction observables                -0.0024
08    GBR + v1 quantum                          -0.0276
08    GBR + v2 quantum                          -0.0364
08    GBR + ZZ interactions                     -0.0138
09    Trained single observable                 -0.0098
09    Trained multi-observable                  -0.0099
10    Per-device VQE energy                     -0.0051
-------------------------------------------------------

Best lift across 9 experiments: -0.0024
Experiments with positive lift: 0/9
Target: +0.1500


## 3. Strategic decision

In [6]:
print("=" * 65)
print("STRATEGIC DECISION")
print("=" * 65)
print("")
print("After 9 quantum experiments across 4 notebooks:")
print("  - Zero experiments showed positive quantum lift")
print("  - Best result: -0.0024 (ZZ interactions, NB08)")
print("  - Worst result: -0.0364 (GBR + v2 quantum, NB08)")
print("")
print("Root cause (confirmed across all experiments):")
print("  Quantum circuits encoding composition features as rotations")
print("  produce nonlinear transforms that tree-based models already")
print("  learn from the raw features. Adding redundant features dilutes")
print("  the model's splits and hurts performance.")
print("")
print("DECISION: Quantum composition encoding is a dead end for this")
print("dataset. Two viable paths forward:")
print("")
print("  PATH A: Ship the ML baseline (tau-b 0.249, p < 1e-10)")
print("    - Strong standalone result")
print("    - Sufficient for lab validation of candidate compositions")
print("    - Reframe milestones: quantum lift is not required for")
print("      the core stability prediction value proposition")
print("")
print("  PATH B: Real quantum chemistry (DFT + VQE on actual fragments)")
print("    - Requires electronic structure calculations (PySCF/Qiskit)")
print("    - Heavy compute, months of work")
print("    - May provide genuine quantum advantage for correlation energy")
print("    - Not appropriate for a validation pilot")
print("")
print("RECOMMENDATION: PATH A — proceed to Phase 2 with ML baseline.")
print("Quantum research continues as a separate R&D track, not a gate.")

STRATEGIC DECISION

After 9 quantum experiments across 4 notebooks:
  - Zero experiments showed positive quantum lift
  - Best result: -0.0024 (ZZ interactions, NB08)
  - Worst result: -0.0364 (GBR + v2 quantum, NB08)

Root cause (confirmed across all experiments):
  Quantum circuits encoding composition features as rotations
  produce nonlinear transforms that tree-based models already
  learn from the raw features. Adding redundant features dilutes
  the model's splits and hurts performance.

DECISION: Quantum composition encoding is a dead end for this
dataset. Two viable paths forward:

  PATH A: Ship the ML baseline (tau-b 0.249, p < 1e-10)
    - Strong standalone result
    - Sufficient for lab validation of candidate compositions
    - Reframe milestones: quantum lift is not required for
      the core stability prediction value proposition

  PATH B: Real quantum chemistry (DFT + VQE on actual fragments)
    - Requires electronic structure calculations (PySCF/Qiskit)
    - Heav

## 4. Milestone re-evaluation with revised framework

In [7]:
print("=" * 65)
print("MILESTONE RE-EVALUATION")
print("=" * 65)
print("")
print("Original milestones (quantum-dependent):")
print("  1. Tau-b lift >= 0.15 (quantum vs ML):     NOT MET (best: -0.0024)")
print("  2. Top-quartile recall >= 15pp:             NOT MET (+1.2pp)")
print("  3. >= 3 compositions >20% gain:             MET (153)")
print("  4. Quantum reduces prediction variance:     NOT MET")
print("  5. Evidence Package reproducible:            MET")
print("  Score: 2/5 — Phase 2 NOT triggered")
print("")
print("After 9 failed quantum experiments, the quantum-dependent")
print("milestones (1, 2, 4) cannot be met with current approach.")
print("")
print("Options:")
print("  A. Keep current milestones → Phase 2 blocked indefinitely")
print("  B. Revise milestones to ML-focused → Phase 2 possible")
print("  C. Pivot to real quantum chemistry → Phase 2 delayed months")
print("")
print("If milestones are revised to ML-focused:")
print("  1. ML tau-b > 0.20 (vs classical baseline):  MET (0.249 vs 0.116)")
print("  2. ML p-value < 0.001:                       MET (p < 1e-10)")
print("  3. >= 3 compositions >20% gain:               MET (153)")
print("  4. Cross-validated stability:                 MET (0.256 ± 0.011)")
print("  5. Evidence Package reproducible:              MET")
print("  Score: 5/5 — Phase 2 TRIGGERED")
print("")
print("NOTE: This is YOUR decision. The data supports the ML baseline.")
print("Quantum remains a research direction, not a prerequisite.")

MILESTONE RE-EVALUATION

Original milestones (quantum-dependent):
  1. Tau-b lift >= 0.15 (quantum vs ML):     NOT MET (best: -0.0024)
  2. Top-quartile recall >= 15pp:             NOT MET (+1.2pp)
  3. >= 3 compositions >20% gain:             MET (153)
  4. Quantum reduces prediction variance:     NOT MET
  5. Evidence Package reproducible:            MET
  Score: 2/5 — Phase 2 NOT triggered

After 9 failed quantum experiments, the quantum-dependent
milestones (1, 2, 4) cannot be met with current approach.

Options:
  A. Keep current milestones → Phase 2 blocked indefinitely
  B. Revise milestones to ML-focused → Phase 2 possible
  C. Pivot to real quantum chemistry → Phase 2 delayed months

If milestones are revised to ML-focused:
  1. ML tau-b > 0.20 (vs classical baseline):  MET (0.249 vs 0.116)
  2. ML p-value < 0.001:                       MET (p < 1e-10)
  3. >= 3 compositions >20% gain:               MET (153)
  4. Cross-validated stability:                 MET (0.256 ± 0.011)


## 5. Nitrogen transfer assessment

In [8]:
print("=" * 65)
print("NITROGEN TRANSFER ASSESSMENT")
print("=" * 65)
print("")
print("The nitrogen liberation engine was predicated on quantum")
print("descriptors transferring from perovskite to N2 fixation.")
print("")
print("Current status:")
print("  - VQE infrastructure works (NB04 shadow test converges)")
print("  - But quantum descriptors add no value to perovskite prediction")
print("  - Transfer of zero-value features is pointless")
print("")
print("Nitrogen assessment:")
print("  - VQE for N2 binding energies is legitimate quantum chemistry")
print("  - But it's a DIFFERENT problem from composition encoding")
print("  - Would require DFT baselines, proper basis sets, real benchmarks")
print("  - Estimated effort: 3-6 months of dedicated quantum chemistry work")
print("")
print("DECISION: Nitrogen stays ON HOLD.")
print("Focus on perovskite ML baseline for Phase 2.")
print("Nitrogen becomes a separate project if/when funded.")

NITROGEN TRANSFER ASSESSMENT

The nitrogen liberation engine was predicated on quantum
descriptors transferring from perovskite to N2 fixation.

Current status:
  - VQE infrastructure works (NB04 shadow test converges)
  - But quantum descriptors add no value to perovskite prediction
  - Transfer of zero-value features is pointless

Nitrogen assessment:
  - VQE for N2 binding energies is legitimate quantum chemistry
  - But it's a DIFFERENT problem from composition encoding
  - Would require DFT baselines, proper basis sets, real benchmarks
  - Estimated effort: 3-6 months of dedicated quantum chemistry work

DECISION: Nitrogen stays ON HOLD.
Focus on perovskite ML baseline for Phase 2.
Nitrogen becomes a separate project if/when funded.


## 6. Final summary

In [9]:
print("=" * 65)
print("NOTEBOOK 10 — FINAL SUMMARY")
print("=" * 65)
print("")
print("Experiments completed: 9 quantum variants across NB02, 08, 09, 10")
print("Positive quantum lift: 0/9")
print(f"Best lift: {best_lift:+.4f} (target: +0.1500)")
print("")
print("ML baseline strength:")
print(f"  Kendall tau-b: {tau_ml:.4f} (p < 1e-10)")
print(f"  MAE: {mae_ml:.4f} log-hours")
print(f"  Cross-validated: 0.256 ± 0.011 (5-fold)")
print(f"  Lift vs classical: +0.133")
print("")
print("Decision point for founder:")
print("  1. Revise milestones to ML-focused → Phase 2 ready NOW")
print("  2. Keep quantum milestones → Phase 2 blocked")
print("  3. Pivot to real quantum chemistry → Phase 2 delayed months")
print("")
print("The honest recommendation: the ML baseline is strong enough")
print("for a lab validation pilot. Quantum is a research bet, not a gate.")

NOTEBOOK 10 — FINAL SUMMARY

Experiments completed: 9 quantum variants across NB02, 08, 09, 10
Positive quantum lift: 0/9
Best lift: -0.0024 (target: +0.1500)

ML baseline strength:
  Kendall tau-b: 0.2490 (p < 1e-10)
  MAE: 1.3053 log-hours
  Cross-validated: 0.256 ± 0.011 (5-fold)
  Lift vs classical: +0.133

Decision point for founder:
  1. Revise milestones to ML-focused → Phase 2 ready NOW
  2. Keep quantum milestones → Phase 2 blocked
  3. Pivot to real quantum chemistry → Phase 2 delayed months

The honest recommendation: the ML baseline is strong enough
for a lab validation pilot. Quantum is a research bet, not a gate.
